# StormEngine V8 — Spatial capacity development

Sigma is fixed at 0.10. Run the three new capacity candidates to validation early stopping, then compare them with the existing converged 64/64 baseline.

In [ ]:
from pathlib import Path
import subprocess, sys
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
DEVICE = 'cuda'
ERA5_ROOT = REPO.parent / 'DownloadDate'
DEV_CACHE = ERA5_ROOT / 'cache' / 'stormengine_dev_2013_2016'
CONFIGS = [
    REPO / 'configs' / 'v8_reconstruction_dev3y_ph096_lat064.yaml',
    REPO / 'configs' / 'v8_reconstruction_dev3y_ph064_lat096.yaml',
    REPO / 'configs' / 'v8_reconstruction_dev3y_ph096_lat096.yaml',
]
OUTPUTS = [
    REPO / 'artifacts' / 'v8_spatial_dev3y_ph096_lat064',
    REPO / 'artifacts' / 'v8_spatial_dev3y_ph064_lat096',
    REPO / 'artifacts' / 'v8_spatial_dev3y_ph096_lat096',
]
BASELINE = REPO / 'artifacts' / 'v8_spatial_dev3y_converged_sigma010'
def run_live(args):
    command = [str(item) for item in args]
    print('Running:', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=REPO, check=True).returncode
print('Repository:', REPO)
print('Development cache:', DEV_CACHE)

In [ ]:
assert (BASELINE / 'develop_summary.json').is_file(), 'The converged 64/64 sigma=0.10 baseline is required.'
run_live([sys.executable, '-u', REPO / 'scripts' / 'build_development_cache.py',
          'verify', '--output-cache', DEV_CACHE])
for config in CONFIGS:
    run_live([sys.executable, '-u', REPO / 'scripts' / 'train_v8_reconstruction.py',
              'preflight', '--device', DEVICE, '--config', config])

In [ ]:
RUN_CAPACITY_CANDIDATES = False
if RUN_CAPACITY_CANDIDATES:
    for config, output in zip(CONFIGS, OUTPUTS):
        summary = output / 'develop_summary.json'
        if summary.is_file():
            print('Already complete:', output)
            continue
        command = [sys.executable, '-u', REPO / 'scripts' / 'train_v8_reconstruction.py',
                   'develop', '--device', DEVICE, '--config', config]
        checkpoint = output / 'last.pt'
        if checkpoint.is_file():
            command += ['--resume', checkpoint]
        run_live(command)

In [ ]:
candidates = [BASELINE, *OUTPUTS]
if all((path / 'develop_summary.json').is_file() for path in candidates):
    run_live([sys.executable, '-u', REPO / 'scripts' / 'compare_v8_spatial_capacity.py',
              *candidates, '--output',
              REPO / 'artifacts' / 'v8_spatial_dev3y_capacity_comparison.json'])
else:
    print('Run all three new capacity candidates before comparison.')

## Stop here

Send the three new develop summaries and histories plus `v8_spatial_dev3y_capacity_comparison.json`. Do not start Processor training yet.